# Train an encoder-decoder model that can convert a date string from one format to another (e.g., from "April 22,2019" to "2019-04-22")

In [1]:
import tokenizers
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
import torchmetrics
import transformers
from torch.utils.data import DataLoader
from datasets import load_dataset
import matplotlib.pyplot as plt
import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd
from sklearn.model_selection import train_test_split

## Steps:
#### 1. Get Data or generate, i'm thinking about finding 1 data set and converting it to another format.
#### 2. We tokenize everything, we can used an alreay made tokenizer, thre is no need for collate cause it has the same size
#### 3. Data Loader
#### 4.  Model building
#### 5. Training

In [2]:
device = 'cuda'

In [3]:
file_path = "CTA_-_Ridership_-_Daily_Boarding_Totals_20250830.csv"

df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "aakcodebreaker/cta-ridership-daily-boarding-totals",
  file_path,
)

print("First 5 records:", df.head())

/tmp/ipykernel_36334/1128755209.py:3: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


First 5 records:   service_date day_type      bus rail_boardings total_rides
0   01/01/2001        U  297,192        126,455     423,647
1   01/02/2001        W  780,827        501,952   1,282,779
2   01/03/2001        W  824,923        536,432   1,361,355
3   01/04/2001        W  870,021        550,011   1,420,032
4   01/05/2001        W  890,426        557,917   1,448,343


In [4]:
df.dtypes

service_date      str
day_type          str
bus               str
rail_boardings    str
total_rides       str
dtype: object

In [5]:
len(df)

8947

In [6]:
df["service_date"] = pd.to_datetime(df["service_date"])
df.dtypes

service_date      datetime64[us]
day_type                     str
bus                          str
rail_boardings               str
total_rides                  str
dtype: object

In [7]:
date_original = df.service_date
date_converted= df["service_date"].dt.strftime("%B %d, %Y")

In [8]:
date_original=date_original.astype("str")
date_original=date_original.reset_index(drop=True)
print(type(date_original))

<class 'pandas.Series'>


In [9]:
date_converted.astype('str')

0       January 01, 2001
1       January 02, 2001
2       January 03, 2001
3       January 04, 2001
4       January 05, 2001
              ...       
8942       June 26, 2025
8943       June 27, 2025
8944       June 28, 2025
8945       June 29, 2025
8946       June 30, 2025
Name: service_date, Length: 8947, dtype: str

In [10]:
date_converted = date_converted.to_numpy()
date_original = date_original.to_numpy()
print(date_converted[1], date_original[1])

January 02, 2001 2001-01-02


In [11]:
train_date,test_date, train_target_date,  test_target_date  = train_test_split(date_original, date_converted, test_size=0.3, random_state=23)

In [12]:
print(len(train_date), len(train_target_date), train_date[1], train_target_date[1])

6262 6262 2022-07-31 July 31, 2022


In [13]:
validation_date,test_date,  validation_target_date,  test_target_date  = train_test_split(test_date, test_target_date, test_size=0.5, random_state=23)

In [14]:
print(len(validation_date), len(validation_target_date), validation_date[1], validation_target_date[1])

1342 1342 2023-03-04 March 04, 2023


In [15]:
print(type(validation_date))

<class 'numpy.ndarray'>


In [16]:
def train_date_to_date():
    for date, target in zip(train_date, train_target_date):
        yield date
        yield target

In [17]:
vocab_size = 100
max_length = 20



date_tokenizer_model = tokenizers.models.BPE(unk_token="<unk>")
date_tokenizer = tokenizers.Tokenizer(date_tokenizer_model)
date_tokenizer.enable_padding(pad_id=0, pad_token="<pad>")
date_tokenizer.enable_truncation(max_length=max_length)
date_tokenizer.pre_tokenizer = tokenizers.pre_tokenizers.Whitespace()
date_tokenizer_trainer = tokenizers.trainers.BpeTrainer(
    vocab_size=vocab_size, special_tokens=["<pad>", "<unk>", "<s>","</s>"]
)
date_tokenizer.train_from_iterator(train_date_to_date(), date_tokenizer_trainer)

In [18]:
date_tokenizer.encode("2001-01-02").ids


[44, 7, 5, 56, 5, 65]

In [19]:
from collections import namedtuple

fields = ["src_token_ids", "src_mask", "tgt_token_ids", "tgt_mask"]
class NmtPair(namedtuple("NmtPairBase", fields)):
    def to(self, device):
        return NmtPair(self.src_token_ids.to(device), self.src_mask.to(device),
                       self.tgt_token_ids.to(device), self.tgt_mask.to(device))


In [20]:
def date_collate_fn(batch):
    src_texts = [pair[0] for pair in batch]
    tgt_texts = [f"<s> {pair[1]} </s>" for pair in batch]
    src_encodings = date_tokenizer.encode_batch(src_texts)
    tgt_encodings = date_tokenizer.encode_batch(tgt_texts)
    src_token_ids = torch.tensor([enc.ids for enc in src_encodings])
    tgt_token_ids = torch.tensor([enc.ids for enc in tgt_encodings])
    src_mask = torch.tensor([enc.attention_mask for enc in src_encodings])
    tgt_mask = torch.tensor([enc.attention_mask for enc in tgt_encodings])
    inputs = NmtPair(src_token_ids, src_mask,
                     tgt_token_ids[:, :-1], tgt_mask[:, :-1])
    labels = tgt_token_ids[:, 1:]
    return inputs, labels

In [21]:
batch_size = 32

train_loader = DataLoader(list(zip(train_date, train_target_date)), batch_size=batch_size,
                              collate_fn=date_collate_fn, shuffle=True)
valid_loader = DataLoader(list(zip(validation_date, validation_target_date)), batch_size=batch_size,
                              collate_fn=date_collate_fn)
test_loader = DataLoader(list(zip(test_date, test_target_date)), batch_size=batch_size,
                             collate_fn=date_collate_fn)


In [22]:
train_loader.dataset[0]

('2017-10-14', 'October 14, 2017')

In [23]:
class DateModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=512, pad_id=0, hidden_dim=512, n_layers=1):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        self.encoder = nn.GRU(embed_dim, hidden_dim, num_layers=n_layers, batch_first=True)
        self.decoder = nn.GRU(embed_dim, hidden_dim, num_layers=n_layers, batch_first=True)
        self.output = nn.Linear(hidden_dim, vocab_size)

    def forward(self, pair):
        src_embeddings = self.embed(pair.src_token_ids)
        tgt_embeddings = self.embed(pair.tgt_token_ids)
        src_lengths = pair.src_mask.sum(dim=1)
        src_packed = pack_padded_sequence(
            src_embeddings, lengths=src_lengths.cpu(),
            batch_first=True, enforce_sorted=False
        )
        _, hidden_states = self.encoder(src_packed)
        outputs, _ = self.decoder(tgt_embeddings, hidden_states)
        return self.output(outputs).permute(0,2,1)

In [24]:
torch.manual_seed(42)
vocab_size = date_tokenizer.get_vocab_size()
date_model = DateModel(vocab_size).to(device)

In [25]:
def evaluate_tm(model, data_loader, metric):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            metric.update(y_pred, y_batch)
    return metric.compute()

def train(model, optimizer, loss_fn, metric, train_loader, valid_loader,
          n_epochs, patience=2, factor=0.5, epoch_callback=None):
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", patience=patience, factor=factor)
    history = {"train_losses": [], "train_metrics": [], "valid_metrics": []}
    for epoch in range(n_epochs):
        total_loss = 0.0
        metric.reset()
        model.train()
        if epoch_callback is not None:
            epoch_callback(model, epoch)
        for index, (X_batch, y_batch) in enumerate(train_loader):
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = loss_fn(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            metric.update(y_pred, y_batch)
            train_metric = metric.compute().item()
            print(f"\rBatch {index + 1}/{len(train_loader)}", end="")
            print(f", loss={total_loss/(index+1):.4f}", end="")
            print(f", {train_metric=:.2%}", end="")
        history["train_losses"].append(total_loss / len(train_loader))
        history["train_metrics"].append(train_metric)
        val_metric = evaluate_tm(model, valid_loader, metric).item()
        history["valid_metrics"].append(val_metric)
        scheduler.step(val_metric)
        print(f"\rEpoch {epoch + 1}/{n_epochs},                      "
              f"train loss: {history['train_losses'][-1]:.4f}, "
              f"train metric: {history['train_metrics'][-1]:.2%}, "
              f"valid metric: {history['valid_metrics'][-1]:.2%}")
    return history


In [27]:
n_epochs = 10
x_entropy = nn.CrossEntropyLoss(ignore_index=0)
optimizer=torch.optim.NAdam(date_model.parameters(), lr = 0.001)
accuracy = torchmetrics.Accuracy(task="multiclass", num_classes = vocab_size, ignore_index=0)
accuracy = accuracy.to(device)
history = train(date_model, optimizer, x_entropy, accuracy, train_loader, valid_loader, n_epochs )

Epoch 1/10,                      train loss: 0.6597, train metric: 80.72%, valid metric: 90.04%
Epoch 2/10,                      train loss: 0.0624, train metric: 98.62%, valid metric: 100.00%
Epoch 3/10,                      train loss: 0.0027, train metric: 100.00%, valid metric: 100.00%
Epoch 4/10,                      train loss: 0.0012, train metric: 100.00%, valid metric: 100.00%
Epoch 5/10,                      train loss: 0.0008, train metric: 100.00%, valid metric: 100.00%
Epoch 6/10,                      train loss: 0.0006, train metric: 100.00%, valid metric: 100.00%
Epoch 7/10,                      train loss: 0.0005, train metric: 100.00%, valid metric: 100.00%
Epoch 8/10,                      train loss: 0.0004, train metric: 100.00%, valid metric: 100.00%
Epoch 9/10,                      train loss: 0.0003, train metric: 100.00%, valid metric: 100.00%
Epoch 10/10,                      train loss: 0.0003, train metric: 100.00%, valid metric: 100.00%


In [28]:
evaluate_tm(date_model, test_loader, accuracy)

tensor(1., device='cuda:0')

In [29]:
def date_convert(model, src_text, max_length=20, pad_id=0, eos_id=3):
    tgt_text = ""
    for index in range(max_length):
        batch, _ = date_collate_fn([(src_text,tgt_text)])
        with torch.no_grad():
            Y_logits = model(batch.to(device))
            Y_token_ids = Y_logits.argmax(dim=1)  # find the best token IDs
            next_token_id = Y_token_ids[0, index]  # take the last token ID

        next_token = date_tokenizer.id_to_token(next_token_id)
        tgt_text += " " + next_token
        if next_token_id == eos_id:
            break
    return tgt_text

In [33]:
def date_convertv2(model, src_text, max_length=20, pad_id=0, eos_id=3):
    generated_ids = []
    for _ in range(max_length):
        decoded_target = date_tokenizer.decode(
                                generated_ids, skip_special_tokens=True
                                )
        batch, _ = date_collate_fn([(src_text, decoded_target)])
        with torch.no_grad():
            logits = model(batch.to(device))
            next_token_id = logits[0, :, -1].argmax().item

        generated_ids.append(next_token_id)

        if next_token_id == eos_id:
            break
    return date_tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    )



In [34]:
date_model.eval()
date_convert(date_model,"2002-05-05")


' May 05 , 200 2 </s>'